<a href="https://colab.research.google.com/github/lucianoselimaj/SigExt/blob/MMD/notebooks/three_layer_agentic_refinement.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title requirements
!pip install transformers datasets accelerate evaluate rouge_score torch jsonlines rake_nltk pytorch_lightning
!pip install rapidfuzz
!pip install boto3
!pip install mistralai
!pip install --upgrade mistralai

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 44.6 MB/s eta 0:00:00
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=8f966237f134623199f5a94af865218a6fffd52ec07a94e4c93c5e9a600da4c3
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.5/140.5 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.0/15.0 MB 59.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 6.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Clone the Repo

In [ ]:
# @markdown check the check-box to clone the repo from sorce, <b>otherwise it will be loaded from shared google drive files</b>

clone_repo = False # @param {type:"boolean"}
if clone_repo:
  !git clone https://github.com/lucianoselimaj/SigExt.git
  path = "/SigExt"

else:
  path = "/content/drive/MyDrive/DNLP-storage/SigExt"


# Preparing Data

In [ ]:
# @title Download requirements for data preparing

%cd {path}

import nltk
nltk.download('stopwords')
nltk.download('punkt_tab')

/content/drive/MyDrive/DNLP-storage/SigExt


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
# @title Run the `prepare_data.py` script
# @markdown Here you can select the dataset you want to use for the training. </br> If you want to use a new dataset, make sure to also change the `path_to_dataset`. </br> No new file will be downloaded or processed if `path_to_dataset` already exists!!

import os
dataset = "cnn" #@param{type:"string"}
path_to_dataset = "experiments/cnn_dataset/" #@param{type:"string"}
if not os.path.exists(path+"/"+path_to_dataset):
  print('New dataset...')
  !python3 src/prepare_data.py --dataset {dataset} --output_dir {path_to_dataset}
  pass

else:
  print(f"Directory already exist. `{path_to_dataset}` will be used")

Directory already exist. `experiments/cnn_dataset/` will be used


# Train

You can modify path to check-points and outputs to load or save different files.
**To keep track of procedure and training routh, insert paths you create in this text down below, under the `paths` ( DO NOT FORGET THE DESCRIPTION :) thank you in advance.  )**



 **⚠️Even for test runs, Change two paths to avoid overwriting existing results⚠️**


---


**Paths**
*   `experiments/cnn_extractor_model/` Baseline code (dataset: cnn),  no modification. output in `experiments/cnn_dataset_with_keyphrase/`
*   `new_path` add you new path here...





In [ ]:

path_to_checkpoint = "experiments/cnn_extractor_model/" #@param{type:"string"}
path_to_output = "experiments/cnn_dataset_with_keyphrase/" #@param{type:"string"}

train_longformer_extractor = False #@param{type:"boolean"}
inference_longformer_extractor = False #@param{type:"boolean"}


In [ ]:
#@title Calling `train_longformer_extractor_context.py` to train the longformer
if train_longformer_extractor:
  !python3 src/train_longformer_extractor_context.py \
    --dataset_dir {path_to_dataset} \
    --checkpoint_dir {path_to_checkpoint}

In [ ]:
# #@title Calling `inference_longformer_extractor.py`
if inference_longformer_extractor:
  !python3 src/inference_longformer_extractor.py \
    --dataset_dir {path_to_dataset} \
    --checkpoint_dir {path_to_checkpoint} \
    --output_dir {path_to_output}

In [ ]:
summerizing_result_path = "experiments/cnn_extsig_predictions_mistral-small-2603_k15_extention2/" #@param{type:"string"}
top_k_kp = 15 #@param{type:"integer"}

In [ ]:
CRITIC_PROMPT_TEMPLATE = """<s>[INST]You are a rigorous data auditor. Your objective is to verify a draft summary against a source text with absolute precision.

Here is the original task, source text, and required key phrases:
<original_task>
{original_task}
</original_task>

Here is the drafted summary:
<summary>
{draft_summary}
</summary>

Evaluate the draft for TWO specific failure modes only:
1. OMITTED KEY PHRASES: Did the draft fail to include any of the required key phrases?
2. FACTUAL DISCREPANCIES: Did the draft include information that contradicts or hallucinates beyond the source text?

STRICT OUTPUT CONSTRAINTS:
- If the draft successfully integrates the key phrases and is factually accurate, output exactly the word: "PASS"
- If there are errors, output ONLY a highly concise, actionable bulleted list of what is missing or incorrect.
- DO NOT include conversational filler, greetings, explanations, or conclusions.
- DO NOT critique grammar, tone, style, or sentence length. Focus strictly on missing facts and key phrases.[/INST]"""
#@title PROMPTS HERE
ENHANCER_PROMPT_TEMPLATE = """<s>[INST]You are a surgical text editor. Your ONLY job is to insert missing key phrases into a draft summary with zero conversational bloat.

### EXAMPLE OF YOUR JOB ###
Draft: "The company reported a profit for the quarter."
Critique: "- Missing key phrase: 'Q3'"
Correct Output: "The company reported a profit for the Q3 quarter."
Incorrect Output (DO NOT DO THIS): "Here is the updated summary: The company reported a profit for the Q3 quarter."
Incorrect Output (DO NOT DO THIS): "The company reported a profit for the quarter, specifically Q3."
### END EXAMPLE ###

Here is the original task, source text, and required key phrases:
<original_task>
{original_task}
</original_task>

Here is the initial draft summary:
<summary>
{draft_summary}
</summary>

Here is the Critique Report:
<critique>
{critique}
</critique>

STRICT EDITORIAL RULES:
1. DO NOT rewrite sentences.
2. DO NOT output conversational filler (e.g., "Here is...", "Updated:").
3. SURGICALLY INSERT the missing keywords from the critique into the existing draft.
4. If the Critique Report says "PASS", output the draft summary exactly as it was provided.

Output ONLY the final summary text.[/INST]"""

In [ ]:
import sys
import os
from types import ModuleType
import time
from mistralai.client import Mistral
def chat(prompt_text, client):
    chat_response = client.chat.complete(
        model="mistral-small-latest", # Recommended to use 'latest' tags for Mistral API
        messages=[{"role": "user", "content": prompt_text}]
    )
    return chat_response.choices[0].message.content

def predict_one_eg_mistral_3_layer(prompt_dict, is_3_layer_active=True):
    client = Mistral(api_key="YdbLhqEepgXav0x8ruTJiWR20GsgWCrD")
    original_task = prompt_dict["prompt_input"]

    # Layer 1: Base Extraction
    draft_summary = ""
    final_output = ""
    for _ in range(5):
        try:
            time.sleep(1.1)
            if is_3_layer_active:
                draft_summary = chat(original_task, client)
            else:
                final_output = chat(original_task, client)
            break
        except Exception as e:
            if "429" in str(e):
                print("L1 Rate limit hit! Sleeping for 10s...")
                time.sleep(2)
    if not is_3_layer_active:

        return final_output or "Error: Layer 1 Failed"

    if not draft_summary: return "Error: Layer 1 Failed"

    # Layer 2: Critic Verification
    critic_prompt = CRITIC_PROMPT_TEMPLATE.format(original_task=original_task, draft_summary=draft_summary)
    critique = ""
    for _ in range(5):
        try:
            time.sleep(1.1)
            critique = chat(critic_prompt, client)
            break
        except Exception as e:
            if "429" in str(e):
                print("L2 Rate limit hit! Sleeping for 10s...")
                time.sleep(2)

    if not critique: return "Error: Layer 2 Failed"
    if len(critique)<10 and "PASS" in critique  :
        return draft_summary
    # Layer 3: Enhancement & Polish
    enhancer_prompt = ENHANCER_PROMPT_TEMPLATE.format(original_task=original_task, draft_summary=draft_summary, critique=critique)
    final_output = ""
    for _ in range(5):
        try:
            time.sleep(1.1)
            final_output = chat(enhancer_prompt, client)
            break
        except Exception as e:
            if "429" in str(e):
                print("L3 Rate limit hit! Sleeping for 10s...")
                time.sleep(2)

    return final_output or "Error: Layer 3 Failed"

class FakePool:
    def __init__(self, processes=None): pass
    def __enter__(self): return self
    def __exit__(self, *args): pass
    def imap(self, func, iterable):
        return map(func, iterable)





import multiprocessing

multiprocessing.Pool = FakePool


m = ModuleType("bedrock_utils")
m.predict_one_eg_mistral = predict_one_eg_mistral_3_layer
m.predict_one_eg_claude_instant = lambda x: "Not configured"
sys.modules["bedrock_utils"] = m


sys.argv = [
    "zs_summarization.py",
    "--model_name", "mistral",
    "--kw_strategy", "sigext_topk",
    "--kw_model_top_k", str(top_k_kp),
    "--dataset", dataset,
    "--dataset_dir", path_to_output,
    "--output_dir", summerizing_result_path
]



sys.path.append('/content/drive/MyDrive/DNLP-storage/SigExt/src')
import zs_summarization
import importlib
importlib.reload(zs_summarization)

zs_summarization.main()

  1%|          | 4/500 [00:31<1:04:36,  7.81s/it]
ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_543/1508722703.py", line 108, in <cell line: 0>
    zs_summarization.main()
  File "/content/drive/MyDrive/DNLP-storage/SigExt/src/zs_summarization.py", line 248, in main
    run_inference(**vars(args))
  File "/content/drive/MyDrive/DNLP-storage/SigExt/src/zs_summarization.py", line 211, in run_inference
    all_res = list(tqdm.tqdm(p.imap(predict_one_eg_fn, inference_data), total=len(inference_data)))
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/tqdm/std.py", line 1181, in __iter__
    for obj in iterable:
               ^^^^^^^^
  File "/tmp/ipykernel_543/1508722703.py", line 43, in predict_one_eg_mistral_3_layer
    time.sleep(1.1)
KeyboardInterrupt

During handling 

TypeError: object of type 'NoneType' has no len()

In [ ]:
import json

def compare_rouge_files(baseline_filepath, verified_filepath):
    """Loads two JSON metric files and prints a side-by-side comparison."""

    try:
        # Load the JSON data from both files
        with open(baseline_filepath, 'r') as f1:
            baseline_metrics = json.load(f1)

        with open(verified_filepath, 'r') as f2:
            verified_metrics = json.load(f2)

    except FileNotFoundError as e:
        print(f"Error loading files: {e}")
        return
    except json.JSONDecodeError as e:
        print(f"Error parsing JSON: {e}")
        return

    # Print the formatted comparison table
    print("=" * 65)
    print(f"{'Metric':<30} {'Baseline':>12} {'Verified':>12} {'Diff':>10}")
    print("=" * 65)

    # A helper function to print each row and calculate the difference
    def print_row(metric_name, key):
        base_val = baseline_metrics.get(key, 0)
        ver_val = verified_metrics.get(key, 0)
        diff = ver_val - base_val
        # Format with a + sign for positive differences
        diff_str = f"{diff:>+10.4f}"
        print(f"{metric_name:<30} {base_val:>12.4f} {ver_val:>12.4f} {diff_str}")

    # Map your desired table labels to the JSON keys
    print_row("ROUGE-1 F1", "rouge1f")
    print_row("ROUGE-L F1", "rougeLf")
    print_row("ROUGE-1 Recall", "rouge1r")
    print_row("ROUGE-2 F1", "rouge2f")
    print_row("ROUGE-Lsum F1", "rougeLsumf")
    print("-" * 65)
    print_row("Generation Length", "gen_len")
    print("=" * 65)

baseline_path = "/content/drive/MyDrive/DNLP-storage/SigExt/experiments/cnn_extsig_predictions_mistral-small-2603_k15/test_metrics.json" #@param{type:"string"}
verified_path = "/content/drive/MyDrive/DNLP-storage/SigExt/experiments/cnn_extsig_predictions_mistral-small-2603_k15_extention2/test_metrics.json" #@param{type:"string"}


# Run the comparison!
compare_rouge_files(baseline_path , verified_path)

Metric                             Baseline     Verified       Diff
ROUGE-1 F1                          38.8941      34.3484    -4.5457
ROUGE-L F1                          24.2784      21.4287    -2.8497
ROUGE-1 Recall                      47.2585      35.6221   -11.6364
ROUGE-2 F1                          14.1088      10.3753    -3.7335
ROUGE-Lsum F1                       33.5527      30.5935    -2.9592
-----------------------------------------------------------------
Generation Length                   77.5900      56.0680   -21.5220
